# GTD Data Preprocessing

This notebook loads the Global Terrorism Database (GTD), cleans the data, and exports it to GeoParquet format for efficient loading in visualizations.

## Data Source
The Global Terrorism Database is maintained by the National Consortium for the Study of Terrorism and Responses to Terrorism (START) at the University of Maryland.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Load the GTD Excel File

In [ ]:
# Define paths
project_dir = Path(os.getcwd()).parent
data_file = project_dir / "Global Terrorism.xlsx"

print(f"Loading data from: {data_file}")
print("This may take a few minutes...")

# Load the Excel file
df_raw = pd.read_excel(
    data_file,
    engine='openpyxl',
    dtype={
        'eventid': 'int64',
        'iyear': 'int16',
        'imonth': 'int8',
        'iday': 'int8',
        'success': 'int8'
    }
)

print(f"\nLoaded {len(df_raw):,} rows and {len(df_raw.columns)} columns")

In [ ]:
# Preview the data
df_raw.head()

In [ ]:
# Check column names
print("All columns:")
for i, col in enumerate(df_raw.columns):
    print(f"{i+1:3d}. {col}")

## 2. Clean Coordinates

In [ ]:
# Check coordinate coverage
print("Coordinate Analysis:")
print(f"Total rows: {len(df_raw):,}")
print(f"Missing latitude: {df_raw['latitude'].isna().sum():,}")
print(f"Missing longitude: {df_raw['longitude'].isna().sum():,}")
print(f"Null Island (0,0): {((df_raw['latitude'] == 0) & (df_raw['longitude'] == 0)).sum():,}")

In [ ]:
# Create working copy
df = df_raw.copy()
initial_count = len(df)

# Remove rows with missing coordinates
df = df.dropna(subset=['latitude', 'longitude'])
print(f"After removing missing coords: {len(df):,} rows")

# Remove Null Island
df = df[~((df['latitude'] == 0) & (df['longitude'] == 0))]
print(f"After removing (0,0): {len(df):,} rows")

# Validate coordinate ranges
df = df[(df['latitude'] >= -90) & (df['latitude'] <= 90)]
df = df[(df['longitude'] >= -180) & (df['longitude'] <= 180)]
print(f"After validating ranges: {len(df):,} rows")

removed = initial_count - len(df)
print(f"\nTotal removed: {removed:,} rows ({100*removed/initial_count:.1f}%)")

## 3. Create DateTime Column

In [ ]:
# Check for 0 values in month/day (indicates unknown)
print("Zero values (unknown dates):")
print(f"imonth == 0: {(df['imonth'] == 0).sum():,}")
print(f"iday == 0: {(df['iday'] == 0).sum():,}")

In [ ]:
# Replace 0 with 1 for unknown month/day
df['imonth'] = df['imonth'].replace(0, 1)
df['iday'] = df['iday'].replace(0, 1)

# Create datetime column with error handling
def safe_date(row):
    try:
        return datetime(int(row['iyear']), int(row['imonth']), int(row['iday']))
    except ValueError:
        try:
            return datetime(int(row['iyear']), int(row['imonth']), 1)
        except ValueError:
            return datetime(int(row['iyear']), 1, 1)

print("Creating datetime column...")
df['event_date'] = df.apply(safe_date, axis=1)
print("Done!")

df['event_date'].describe()

## 4. Select Essential Columns

In [ ]:
essential_columns = [
    'eventid',
    'iyear',
    'imonth',
    'iday',
    'event_date',
    'country_txt',
    'region_txt',
    'city',
    'latitude',
    'longitude',
    'attacktype1_txt',
    'targtype1_txt',
    'gname',
    'nkill',
    'nwound',
    'success',
    'weaptype1_txt',
    'summary'
]

# Check which columns exist
available = [col for col in essential_columns if col in df.columns]
missing = [col for col in essential_columns if col not in df.columns]

print(f"Available columns: {len(available)}")
if missing:
    print(f"Missing columns: {missing}")

df = df[available].copy()
print(f"\nFinal columns: {len(df.columns)}")

## 5. Handle Missing Values

In [ ]:
# Check missing values
print("Missing values per column:")
missing_counts = df.isna().sum()
for col, count in missing_counts.items():
    if count > 0:
        pct = 100 * count / len(df)
        print(f"  {col}: {count:,} ({pct:.1f}%)")

In [ ]:
# Fill casualty NaN with 0
df['nkill'] = df['nkill'].fillna(0).astype('float32')
df['nwound'] = df['nwound'].fillna(0).astype('float32')

# Create total casualties column
df['total_casualties'] = df['nkill'] + df['nwound']

print("Casualty statistics:")
print(f"Total killed: {df['nkill'].sum():,.0f}")
print(f"Total wounded: {df['nwound'].sum():,.0f}")
print(f"Total casualties: {df['total_casualties'].sum():,.0f}")

## 6. Optimize Data Types

In [ ]:
# Optimize string columns
if 'city' in df.columns:
    df['city'] = df['city'].astype('string')
if 'gname' in df.columns:
    df['gname'] = df['gname'].astype('string')
if 'summary' in df.columns:
    df['summary'] = df['summary'].astype('string')

# Convert to category for categorical columns
categorical_cols = ['country_txt', 'region_txt', 'attacktype1_txt', 'targtype1_txt', 'weaptype1_txt']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("Data types:")
print(df.dtypes)

In [ ]:
# Check memory usage
memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"DataFrame memory usage: {memory_mb:.1f} MB")

## 7. Data Quality Report

In [ ]:
print("="*60)
print("GTD DATA QUALITY REPORT")
print("="*60)

print(f"\nTotal Records: {len(df):,}")

print("\n" + "-"*40)
print("TEMPORAL DISTRIBUTION")
print("-"*40)
print(f"Year Range: {df['iyear'].min()} - {df['iyear'].max()}")
print(f"Years with data: {df['iyear'].nunique()}")

print("\n" + "-"*40)
print("GEOGRAPHIC COVERAGE")
print("-"*40)
print(f"Countries: {df['country_txt'].nunique()}")
print(f"Regions: {df['region_txt'].nunique()}")

print("\nIncidents by Region:")
for region, count in df['region_txt'].value_counts().items():
    print(f"  {region}: {count:,}")

print("\n" + "-"*40)
print("ATTACK TYPES")
print("-"*40)
for attack, count in df['attacktype1_txt'].value_counts().items():
    print(f"  {attack}: {count:,}")

In [ ]:
# Visualize temporal distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Incidents per year
yearly_counts = df['iyear'].value_counts().sort_index()
axes[0].bar(yearly_counts.index, yearly_counts.values, color='#e41a1c', alpha=0.7)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Incidents')
axes[0].set_title('Terrorist Incidents per Year')

# Incidents by region
region_counts = df['region_txt'].value_counts()
axes[1].barh(region_counts.index, region_counts.values, color='#377eb8', alpha=0.7)
axes[1].set_xlabel('Number of Incidents')
axes[1].set_title('Incidents by Region')

plt.tight_layout()
plt.show()

## 8. Export to Parquet

In [ ]:
# Define output path
output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "gtd_processed.parquet"

# Export to Parquet
print(f"Exporting to: {output_file}")
df.to_parquet(
    output_file,
    engine='pyarrow',
    compression='snappy',
    index=False
)

# Check file size
file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"\nParquet file size: {file_size_mb:.1f} MB")

print("\nPreprocessing complete!")

In [ ]:
# Verify the exported file
df_check = pd.read_parquet(output_file)
print(f"Verification - Loaded {len(df_check):,} rows")
df_check.head()